In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
BASE_DIR = r"C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-FMNIST\prune_layers_ALL"

BATCH_SIZES = [64, 1024, 60000]

PRUNING_PERCENTAGES = [
    0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7,
    0.8, 0.82, 0.84, 0.86, 0.88, 0.9,
    0.92, 0.94, 0.96, 0.98, 1.0
]

# =========================
# OUTPUT
# =========================
AUC_GRAPH_DIR = os.path.join(BASE_DIR, "AUC_graph")
AUC_DATA_DIR = os.path.join(BASE_DIR, "AUC_data")
os.makedirs(AUC_GRAPH_DIR, exist_ok=True)
os.makedirs(AUC_DATA_DIR, exist_ok=True)

# =========================
# STYLE (MATCH CNN)
# =========================
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 10
})

CE_MAX = np.log(10)

# =========================
# COLOR MAP (SAME AS BEFORE)
# =========================
PRUNING_COLOR_MAP = {
    0.0:   "#17becf",
    0.1:   "#B9D9EB",
    0.2:   "#bcbd22",
    0.3:   "#7f7f7f",
    0.4:   "#e377c2",
    0.5:   "#8c564b",
    0.6:   "#800080",
    0.7:   "#d62728",
    0.8:   "#2ca02c",
    0.82:  "#66c2a5",
    0.84:  "#fc8d62",
    0.86:  "#8da0cb",
    0.88:  "#e78ac3",
    0.9:   "#C5B0D5",
    0.92:  "#8DD3C7",
    0.94:  "#BEBADA",
    0.96:  "#80B1D3",
    0.98:  "#FDB462",
    1.0:   "#1f77b4"
}

# =========================
# HELPERS (MATCH CNN)
# =========================
def label_from_p(p):
    pct = p * 100
    if abs(pct - round(pct)) < 1e-9:
        return f"P%={int(round(pct))}"
    return f"P%={pct:g}"

all_auc_records = []

# =========================
# MAIN LOOP
# =========================
for bs in BATCH_SIZES:
    print(f"\nProcessing BS={bs}")

    avg_dfs = {}

    # -------------------------
    # LOAD FILES
    # -------------------------
    for p in PRUNING_PERCENTAGES:
        file_path = os.path.join(
            BASE_DIR,
            f"p-percentage_{p}",
            f"batch_size_{bs}",
            f"averaged_runs_p_{p}_bs_{bs}.csv"
        )

        if not os.path.isfile(file_path):
            print(f"[WARNING] Missing file: {file_path}")
            continue

        avg_dfs[p] = pd.read_csv(file_path)

    if not avg_dfs:
        print(f"[WARNING] No data for BS={bs}")
        continue

    # -------------------------
    # ALIGN BATCH RANGE
    # -------------------------
    non100 = {
        p: df for p, df in avg_dfs.items()
        if p != 1.0 and "Batch_Number" in df.columns
    }
    non100 = {p: df for p, df in non100.items() if not df.empty}

    if non100:
        lowest_max_batch = min(df["Batch_Number"].max() for df in non100.values())
    else:
        lowest_max_batch = min(df["Batch_Number"].max() for df in avg_dfs.values())

    # -------------------------
    # PLOT FUNCTION (CNN STYLE)
    # -------------------------
    def plot_and_save_ce(ce_column, y_label, file_stub):

        plt.figure(figsize=(10, 5))  # MATCH CNN SIZE
        plt.title(f"SLP | MNIST | BS={bs}")
        plt.xlabel("Batch Number")
        plt.ylabel(y_label)

        plt.xlim(0, lowest_max_batch)
        plt.xticks(np.arange(0, int(lowest_max_batch) + 1, 50))
        plt.yticks(np.arange(0, 2.5, 0.5))
        plt.ylim(0, 2.5)

        # ln(10) line label
        plt.text(
            0.40,
            CE_MAX,
            r"$\ln(10)$",
            transform=plt.gca().get_yaxis_transform(),
            fontsize=12,
            va="center",
            ha="left",
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=1)
        )

        for p in sorted(avg_dfs.keys(), reverse=True):
            df = avg_dfs[p].copy()
            df = df[df["Batch_Number"] <= lowest_max_batch]

            color = PRUNING_COLOR_MAP.get(p, "#000000")

            if p == 1.0:
                ce_value = min(df[ce_column].iloc[-1], CE_MAX)
                x = np.array([0, lowest_max_batch])
                y = np.array([ce_value, ce_value])
                auc = ce_value * lowest_max_batch

                plt.plot(
                    x, y,
                    color=color,
                    linewidth=2.5,
                    label=f"{label_from_p(p)} | AUC={auc:.2f}"
                )

            else:
                x = df["Batch_Number"].to_numpy()
                y = np.minimum(df[ce_column].to_numpy(), CE_MAX)

                # REMOVE NaNs (IMPORTANT)
                mask = np.isfinite(x) & np.isfinite(y)
                x, y = x[mask], y[mask]

                if len(x) == 0:
                    print(f"[WARNING] No valid data for P%={p}")
                    continue

                auc = np.trapezoid(y, x)

                plt.plot(
                    x, y,
                    color=color,
                    linewidth=2.0,
                    label=f"{label_from_p(p)} | AUC={auc:.2f}"
                )
                plt.fill_between(x, y, alpha=0.2, color=color)

            all_auc_records.append([bs, p, ce_column, auc])

        plt.grid(True)
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
        plt.tight_layout()

        fig_path = os.path.join(AUC_GRAPH_DIR, f"{file_stub}_BS_{bs}.png")
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        plt.show()
        plt.close()

        print(f"[SAVED] {fig_path}")

    # -------------------------
    # RUN PLOTS
    # -------------------------
    plot_and_save_ce("Avg_CE_Train", "Average CE Train", "Avg_CE_Train")
    plot_and_save_ce("Avg_CE_Test", "Average CE Test", "Avg_CE_Test")

# =========================
# SAVE CSV
# =========================
if all_auc_records:
    auc_df = pd.DataFrame(
        all_auc_records,
        columns=["Batch_Size", "Pruning_Percentage", "CE_Type", "AUC"]
    )
    csv_path = os.path.join(AUC_DATA_DIR, "AUC_ALL_BS.csv")
    auc_df.to_csv(csv_path, index=False)
    print(f"[SAVED] {csv_path}")

print("\n✅ All SLP AUC graphs and data saved successfully.")

KeyError: 'PolyCollection:kwdoc'